In [4]:
import json
from pathlib import Path
import pandas as pd
from tqdm import tqdm

Download benchmark files

In [ ]:
%%bash

data_dir="data/trec_pm2019"
mkdir -p "$data_dir"
cd "$data_dir"
wget https://trec.nist.gov/data/precmed/topics2019.xml
wget https://trec.nist.gov/data/precmed/qrels-treceval-trials.38.txt

--2026-09-21 14:47:00--  https://trec.nist.gov/data/precmed/topics2019.xml
Resolving trec.nist.gov (trec.nist.gov)... 198.18.1.226
Connecting to trec.nist.gov (trec.nist.gov)|198.18.1.226|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [application/xml]
Saving to: ‘topics2019.xml.1’

     0K ......                                                 39.2M=0s

2026-09-21 14:47:02 (39.2 MB/s) - ‘topics2019.xml.1’ saved [6302]

--2026-09-21 14:47:02--  https://trec.nist.gov/data/precmed/qrels-treceval-trials.38.txt
Resolving trec.nist.gov (trec.nist.gov)... 198.18.1.226
Connecting to trec.nist.gov (trec.nist.gov)|198.18.1.226|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/plain]
Saving to: ‘qrels-treceval-trials.38.txt’

     0K .......... .......... .......... .......... .......... 41.0K
    50K .......... .......... .......... .......... .......... 60.7K
   100K .......... .......... .......... .......... ...

Download trial corpus

In [3]:
%%bash
set -e

data_dir="data/trec_pm2019"
mkdir -p "$data_dir"
cd "$data_dir"

for part in 0 1 2 3; do
    wget -c "https://trec-cds.org/clinical_trials.${part}.tar.gz"
done

--2026-09-28 16:17:19--  https://trec-cds.org/clinical_trials.0.tar.gz
Resolving trec-cds.org (trec-cds.org)... 198.18.0.218
Connecting to trec-cds.org (trec-cds.org)|198.18.0.218|:443... connected.
HTTP request sent, awaiting response... 416 Requested Range Not Satisfiable

    The file is already fully retrieved; nothing to do.

--2026-09-28 16:17:20--  https://trec-cds.org/clinical_trials.1.tar.gz
Resolving trec-cds.org (trec-cds.org)... 198.18.0.218
Connecting to trec-cds.org (trec-cds.org)|198.18.0.218|:443... connected.
HTTP request sent, awaiting response... 416 Requested Range Not Satisfiable

    The file is already fully retrieved; nothing to do.

--2026-09-28 16:17:20--  https://trec-cds.org/clinical_trials.2.tar.gz
Resolving trec-cds.org (trec-cds.org)... 198.18.0.218
Connecting to trec-cds.org (trec-cds.org)|198.18.0.218|:443... connected.
HTTP request sent, awaiting response... 416 Requested Range Not Satisfiable

    The file is already fully retrieved; nothing to do.

-

Extract demo data

In [7]:
import shutil
import tarfile

data_dir = Path("data/trec_pm2019")
qrels_path = data_dir / "qrels-treceval-trials.38.txt"
selected_dir = data_dir / "selected_trials"
selected_dir.mkdir(parents=True, exist_ok=True)

qrels = pd.read_csv(
    qrels_path,
    sep=r"\s+",
    header=None,
    names=["topic_id", "unused", "nct_id", "relevance"],
)

target_ids = set(qrels["nct_id"].dropna().astype(str).unique())

print("Judgments:", len(qrels))
print("Target NCT IDs:", len(target_ids))

Judgments: 12996
Target NCT IDs: 8567


In [5]:
found_ids = {
    path.stem
    for path in selected_dir.glob("NCT*.xml")
}

archive_paths = [
    data_dir / f"clinical_trials.{part}.tar.gz"
    for part in range(4)
]

for archive_path in archive_paths:
    print(f"Scanning {archive_path.name}")

    with tarfile.open(archive_path, mode="r:gz") as archive:
        for member in tqdm(
            archive,
            desc=archive_path.name,
            unit="file",
        ):
            if not member.isfile():
                continue

            member_name = Path(member.name)
            if member_name.suffix.lower() != ".xml":
                continue

            nct_id = member_name.stem

            if nct_id not in target_ids or nct_id in found_ids:
                continue

            source = archive.extractfile(member)
            if source is None:
                continue

            destination = selected_dir / f"{nct_id}.xml"
            temporary = selected_dir / f".{nct_id}.xml.part"

            with source, temporary.open("wb") as output:
                shutil.copyfileobj(source, output)

            temporary.replace(destination)
            found_ids.add(nct_id)

    print(
        f"Found so far: {len(found_ids)} / {len(target_ids)}"
    )

Scanning clinical_trials.0.tar.gz


clinical_trials.0.tar.gz: 79860file [00:12, 6213.35file/s]


Found so far: 2249 / 8567
Scanning clinical_trials.1.tar.gz


clinical_trials.1.tar.gz: 76360file [00:12, 6256.06file/s]


Found so far: 4363 / 8567
Scanning clinical_trials.2.tar.gz


clinical_trials.2.tar.gz: 76726file [00:11, 6958.69file/s]


Found so far: 6510 / 8567
Scanning clinical_trials.3.tar.gz


clinical_trials.3.tar.gz: 73692file [00:09, 7418.03file/s]

Found so far: 8567 / 8567


Convert to markdown files

In [8]:
import xml.etree.ElementTree as ET

xml_dir = Path("data/trec_pm2019/selected_trials")
md_dir = Path("data/trec_pm2019/trial_markdown")
md_dir.mkdir(parents=True, exist_ok=True)


def convert_xml_to_md(xml_path):
    root = ET.parse(xml_path).getroot()

    def text(xpath):
        node = root.find(xpath)
        if node is None:
            return ""
        return " ".join("".join(node.itertext()).split())

    def texts(xpath):
        return [
            " ".join("".join(node.itertext()).split())
            for node in root.findall(xpath)
        ]

    nct_id = text("id_info/nct_id")
    title = text("brief_title")

    sections = [
        ("Title", title),
        ("NCT ID", nct_id),
        ("Conditions", ", ".join(texts("condition"))),
        (
            "Interventions",
            ", ".join(
                texts("intervention/intervention_name")
            ),
        ),
        ("Brief Summary", text("brief_summary/textblock")),
        (
            "Detailed Description",
            text("detailed_description/textblock"),
        ),
        ("Gender", text("eligibility/gender")),
        ("Minimum Age", text("eligibility/minimum_age")),
        ("Maximum Age", text("eligibility/maximum_age")),
        (
            "Eligibility Criteria",
            text("eligibility/criteria/textblock"),
        ),
    ]

    markdown = "\n\n".join(
        f"## {heading}\n\n{content}"
        for heading, content in sections
        if content
    )

    output_path = md_dir / f"{nct_id}.md"
    output_path.write_text(markdown + "\n", encoding="utf-8")


for xml_path in tqdm(
    sorted(xml_dir.glob("NCT*.xml")),
    desc="Converting XML to Markdown",
):
    convert_xml_to_md(xml_path)

Converting XML to Markdown: 100%|██████████| 8567/8567 [00:10<00:00, 831.01it/s] 
